# 23 · GSE135779 · scRNA_seq · normalise

Reads `pseudobulk_counts.rds`, NCBI gene_info. Writes `data/run_artifacts/GSE135779/expression.rds`.

1. **Ensembl → symbol**: Ensembl identifier → NCBI Entrez → current NCBI symbol (gene_info), the same
   names as the other two studies. Genes NCBI does not map keep the symbol in the GEO gene list.
   Rows sharing a symbol are summed.
2. **Filter low counts** (WGCNA FAQ): remove genes with a count below 10 in more than 90% of samples.
3. **PFlog1pPF** (Booeshaghi et al., bioRxiv 2022): PF, log1p, PF, as in notebook 12.

In [1]:
source("../src/paths.R")
pb <- readRDS(art("GSE135779", "pseudobulk_counts.rds"))
gi <- read.delim(file.path(NCBI, "Homo_sapiens.gene_info.gz"), quote = "", colClasses = "character")
ens <- regmatches(gi$dbXrefs, regexpr("ENSG[0-9]+", gi$dbXrefs))
ens_map <- setNames(gi$Symbol[grepl("ENSG[0-9]+", gi$dbXrefs)], ens)
sym <- ens_map[pb$genes$ensembl]
c(genes = nrow(pb$genes), mapped_by_ncbi = sum(!is.na(sym)))
sym[is.na(sym)] <- pb$genes$symbol[is.na(sym)]
counts <- rowsum(pb$counts, sym)
c(genes_after_merging_shared_symbols = nrow(counts))

genes mapped_by_ncbi 
         32738          22831

genes_after_merging_shared_symbols 
                             32614

**Result.** NCBI maps 22,831 of 32,738 Ensembl identifiers; after merging shared symbols, 32,614
genes remain.

In [2]:
keep <- rowMeans(counts < 10) <= 0.90
c(genes_before = nrow(counts), genes_kept = sum(keep))
counts <- counts[keep, ]
pf <- function(X) t(t(X) / colSums(X) * mean(colSums(X)))
E  <- pf(log1p(pf(counts)))
range(colSums(E))
saveRDS(list(E = E, counts = counts, meta = pb$meta), art("GSE135779", "expression.rds"))

genes_before   genes_kept 
       32614        15353

[1] 76573.67 76573.67

**Result.** 15,353 genes pass the filter. After PFlog1pPF every sample has the same total.